# Riesgo de crédito en un banco
## Notebook 03 · Análisis exploratorio del riesgo

**Entrada:** tabla `prestamos_variables` de `data/banco.db`, generada en `02_variables_sql.ipynb`.

**Qué hace este notebook:** compara la tasa de mora entre grupos de préstamos para identificar los factores de riesgo, antes de construir un modelo.

**Conceptos estadísticos:**
- **Tasa de mora de un grupo** = probabilidad condicional P(malo | grupo).
- **Lift** = cuántas veces más (o menos) riesgo tiene un grupo que el promedio.
- **Intervalo de confianza:** con pocos préstamos, una tasa puede variar mucho por azar. El intervalo muestra ese margen.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

conexion = sqlite3.connect("../data/banco.db")

def consulta(sql):
    return pd.read_sql(sql, conexion)

# Columnas de las tablas del cliente y del distrito (para revisar sus nombres)
print("client:", consulta("SELECT * FROM client LIMIT 1").columns.tolist())
print("district:", consulta("SELECT * FROM district LIMIT 1").columns.tolist())

client: ['client_id', 'gender', 'birth_date', 'district_id']
district: ['district_id', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']


In [2]:
df = consulta("""
SELECT
    v.*,
    c.gender                                                AS genero,
    ROUND((julianday(v.fecha_prestamo) - julianday(c.birth_date)) / 365.25, 0) AS edad,
    d.A11                                                   AS salario_promedio_distrito,
    d.A13                                                   AS desempleo_distrito
FROM prestamos_variables AS v
JOIN disp AS dp
    ON  dp.account_id = v.account_id
    AND dp.type = 'Titular'
JOIN client AS c
    ON c.client_id = dp.client_id
JOIN district AS d
    ON d.district_id = c.district_id
""")

assert len(df) == 682, "El JOIN duplicó o perdió préstamos"

tasa_global = df["malo"].mean()
print(f"Préstamos: {len(df)} | Tasa de mora global: {tasa_global:.1%}")

Préstamos: 682 | Tasa de mora global: 11.1%


---
## 1. Tasa de mora por grupo

Cada variable numérica se divide en **cuartiles** (cuatro grupos con el mismo número de préstamos) y se calcula la tasa de mora de cada uno.

Para cada tasa se calcula un **intervalo de confianza del 95% (método de Wilson)**. Con grupos de unos 170 préstamos y pocos casos de mora, una diferencia de 2 o 3 puntos puede deberse solo al azar. Si los intervalos de dos grupos se superponen mucho, no se puede afirmar que su riesgo sea distinto.

In [3]:
def intervalo_wilson(malos, n, z=1.96):
    """Intervalo de confianza del 95% para una proporción (método de Wilson)."""
    p = malos / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    margen = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centro - margen, centro + margen


def tasa_por_grupo(datos, columna, cuantiles=4):
    """Tasa de mora, intervalo de confianza y lift por grupo de una variable."""
    grupos = pd.qcut(datos[columna], q=cuantiles, duplicates="drop")
    tabla = datos.groupby(grupos, observed=True)["malo"].agg(prestamos="count", malos="sum")
    tabla["tasa"] = tabla["malos"] / tabla["prestamos"]
    tabla[["ic_inf", "ic_sup"]] = [
        intervalo_wilson(m, n) for m, n in zip(tabla["malos"], tabla["prestamos"])
    ]
    tabla["lift"] = tabla["tasa"] / tasa_global
    return tabla.round(3)